# Notebook 9 | Bayes' Theorem

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## Bayes' theorem

Bayes' theorem inverts a conditional by renormalizing the product-rule piece with the law of total probability.
For generic random variables $X$ and $Y$ with values $x$ and $y$, in full form first and then in shorthand,

$$p(X = x \mid Y = y) = \frac{p(Y = y \mid X = x) \, p(X = x)}{\sum_{x'} p(Y = y \mid X = x') \, p(X = x')},$$
$$p(x \mid y) = \frac{p(y \mid x) \, p(x)}{\sum_{x'} p(y \mid x') \, p(x')}.$$

## Derivation

1. Write the posterior with the product rule: $p(b \mid c) = p(b, c) / p(c) = p(b) \, p(c \mid b) / p(c)$.
2. Expand the denominator with the law of total probability: $p(c) = \sum_{b'} p(b') \, p(c \mid b')$.
3. Substitute back to get $p(b \mid c) = p(b) \, p(c \mid b) / \sum_{b'} p(b') \, p(c \mid b')$.
4. The same pattern holds for two signals: condition on $(h, c)$ and use $p(h, c \mid b) = p(h \mid b) \, p(c \mid b)$.

## Worked example (by hand)

Single signal, black paint:

$$p(\text{Porsche} \mid \text{black}) = \frac{0.12}{0.31} \approx 0.387.$$

Single signal, 700 hp (only Porsche reaches 700 hp):

$$p(\text{Porsche} \mid 700) = 1.$$

Two signals, 600 hp and black, via conditional independence:

$$p(\text{Porsche} \mid 600, \text{black}) = \frac{0.006}{0 + 0.006 + 0.2 \cdot 0.3 \cdot 0.2} = \frac{0.006}{0.018} = \frac{1}{3},$$

while $p(\text{Ferrari} \mid 600, \text{black}) = 2/3$: the 600-hp black car is more likely a Ferrari despite the lower prior.

In [2]:
import math

import pandas as pd

joint = create_joint_car_distribution()
idx = pd.IndexSlice

# single signal: p(Porsche | black) = 0.12 / 0.31
p_porsche_black = joint.loc[idx["Porsche", :, "black"],].sum()
p_black = joint.xs("black", level="color").sum()
assert math.isclose(p_black, 0.31)
assert math.isclose(p_porsche_black / p_black, 0.12 / 0.31)
assert math.isclose(p_porsche_black / p_black, 0.387, rel_tol=1e-3)

# single signal: p(Porsche | 700) = 1 since only Porsche reaches 700 hp
p_porsche_700 = joint.loc[idx["Porsche", 700, :],].sum()
p_700 = joint.xs(700, level="horsepower").sum()
assert math.isclose(p_700, 0.015)
assert math.isclose(p_porsche_700 / p_700, 1.0)

# two signals: p(Porsche | 600, black) = 0.006 / 0.018 = 1/3, p(Ferrari | 600, black) = 2/3
numerator = 0.3 * 0.05 * 0.4
denominator = 0.0 + 0.3 * 0.05 * 0.4 + 0.2 * 0.3 * 0.2
assert math.isclose(numerator, 0.006)
assert math.isclose(denominator, 0.018)
assert math.isclose(numerator / denominator, 1 / 3)
assert math.isclose((0.2 * 0.3 * 0.2) / denominator, 2 / 3)
numerator / denominator

0.3333333333333333

## Generalization

Slicing the joint on any observed signal and renormalizing gives the full posterior over brands.
Below, conditioning on 600 hp alone already favors Ferrari, and adding black paint keeps the 1:2 odds.

In [3]:
# full posterior over brands given 600 hp: p(b | 600) = p(b, 600) / p(600)
posterior_600 = joint.xs(600, level="horsepower").groupby(level="brand").sum()
posterior_600 = posterior_600 / posterior_600.sum()
assert math.isclose(posterior_600.sum(), 1.0)
assert math.isclose(posterior_600["Porsche"], 0.015 / 0.075)
posterior_600

brand
Ferrari    0.8
Porsche    0.2
Name: p, dtype: float64

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Probability" (covers Bayes' rule), https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.